# Teste manual — Gold (`GoldOrchestrator`)

Pré-requisito: silver populada (`python run_silver.py daily` ou `one feriados`).

Fluxo: **silver** → `read_silver` / `materialize` → valor gold (SQL-ready ou builder).

O pipeline gold (`run_gold.py`, futuro) decide quando persistir no SQLite.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Abra o notebook na raiz do repo ou em notebooks/")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import get_settings
from pipelines.gold import BUILDER_NAMES, GoldOrchestrator

settings = get_settings()
orch = GoldOrchestrator()
START, END = "2026-01-01", "2026-01-31"

print("SILVER:", settings.silver_root)
print("Builders:", BUILDER_NAMES)

SILVER: \\BRZ1WS2478\Brazil_Dept\Mesa_Papel\Chila\projetos\brazil_fixed_income_analytics\data\silver
Builders: ('feriados', 'cdi', 'ptax', 'bmf', 'anbimas', 'ipca_dict', 'vna_lft')


## 1. Feriados (materializer — SQL-ready)

Silver: `feriados/snapshot=1` (leitura integral, sem datas). Saída: `list[str]` ISO para tabela `FERIADOS`.

In [2]:
silver_feriados = orch.read_feriados()
display(silver_feriados["feriados"].head())

result_feriados = orch.materialize_feriados()
datas = result_feriados.value
print("total feriados:", len(datas))
print("primeiros:", datas[:5])
print("ultimos:", datas[-5:])

,data
0,2001-01-01
1,2001-02-26
2,2001-02-27
3,2001-04-13
4,2001-04-21


total feriados: 1263
primeiros: ['2001-01-01', '2001-02-26', '2001-02-27', '2001-04-13', '2001-04-21']
ultimos: ['2099-10-12', '2099-11-02', '2099-11-15', '2099-11-20', '2099-12-25']


In [3]:
silver_feriados

{'feriados':             data
 0     2001-01-01
 1     2001-02-26
 2     2001-02-27
 3     2001-04-13
 4     2001-04-21
 ...          ...
 1258  2099-10-12
 1259  2099-11-02
 1260  2099-11-15
 1261  2099-11-20
 1262  2099-12-25
 
 [1263 rows x 1 columns]}

## 2. CDI (materializer — SQL-ready)

Silver: partições `cdi/data=YYYY-MM-DD`. Passe a lista de datas (o pipeline define quais carregar).

In [2]:
dates = ["2026-01-15", "2026-01-16"]  # exemplo; em produção o pipeline passa a lista
result_cdi = orch.materialize_cdi(dates)
display(result_cdi.value)
print("linhas:", len(result_cdi.value))

,data_referencia,cdi
0,2026-01-15,14.9
1,2026-01-16,14.9


linhas: 2


## 3. PTAX USD (materializer — SQL-ready)

Silver: partições `ptax/data=YYYY-MM-DD`. Passe a lista de datas (o pipeline define quais carregar).

In [2]:
dates_ptax = ["2026-01-15", "2026-01-16"]  # exemplo; em produção o pipeline passa a lista
result_ptax = orch.materialize_ptax(dates_ptax)
display(result_ptax.value)
print("linhas:", len(result_ptax.value))

,data_referencia,ptax_compra,ptax_venda
0,2026-01-15,5.384,5.3846
1,2026-01-16,5.3792,5.3798


linhas: 2


## 4. BMF ajustes (materializer — SQL-ready)

Silver: partições `ajustes_bmf/data=YYYY-MM-DD`. Passe a lista de datas.

In [2]:
dates_bmf = ["2026-01-15", "2026-01-16"]  # exemplo; em produção o pipeline passa a lista
result_bmf = orch.materialize_bmf(dates_bmf)
display(result_bmf.value)
print("linhas:", len(result_bmf.value))

,data_referencia,data_vencimento,ticker,taxa_ajuste,quantidade_ajuste,codigo_isin
0,2026-01-15,2026-01-15,DAPF26,0.000,100000.00,BRBMEFDAP4M4
1,2026-01-15,2026-02-02,DI1G26,14.894,99341.04,BRBMEFD1I8F3
2,2026-01-15,2026-02-18,DAPG26,9.594,99203.40,BRBMEFDAP553
3,2026-01-15,2026-03-02,DI1H26,14.869,98363.28,BRBMEFD1I8G1
4,2026-01-15,2026-03-16,DAPH26,7.786,98816.93,BRBMEFDAP579
...,...,...,...,...,...,...
124,2026-01-16,2041-01-02,DI1F41,13.677,14881.38,BRBMEFD1I8T4
125,2026-01-16,2045-05-15,DAPK45,7.275,25963.05,BRBMEFDAP3B9
126,2026-01-16,2050-08-15,DAPQ50,7.210,18265.64,BRBMEFDAP3C7
127,2026-01-16,2055-05-17,DAPK55,7.155,13349.68,BRBMEFDAP3D5


linhas: 129


## 5. Mercado secundário (materializer — SQL-ready)

Silver: partições `mercado_secundario/data=YYYY-MM-DD`. Passe a lista de datas.

In [ ]:
dates_ms = ["2026-01-15", "2026-01-16"]
result_ms = orch.materialize_mercado_secundario(dates_ms)
display(result_ms.value)
print("linhas:", len(result_ms.value))

data_referencia        str
cdi                float64
dtype: object

In [ ]:
dates_liq = ["2026-01-15", "2026-01-16"]
result_liq = orch.materialize_liquidacoes_mercado(dates_liq)
display(result_liq.value)
print("linhas:", len(result_liq.value))

## 5. Outros (descomente quando implementados)

| Nome | Silver | Saída |
|------|--------|-------|
| anbimas | mercado_secundario | dict DataFrame |
| ipca_dict | ipca_indice, projecoes | dict |
| vna_lft | (futuro) | TBD |

In [ ]:
# for name in ("ipca_dict",):
#     out = orch.materialize(name, START, END)
#     print(name, type(out.value))